# 01 - Exploratory Data Analysis

In [1]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import plotly.express as px
from src.data_loader import load_cities, load_salaries
pd.set_option('display.max_columns', None)

## Descripcion General del dataset


In [2]:
import pandas as pd
from pathlib import Path

df = pd.read_csv('../data/processed/cities_processed.csv')

print(f"Total ciudades: {len(df)}")
print(f"Países únicos: {df['country'].nunique()}")
print(f"Regiones únicas: {df['region'].nunique()}")
print(f"\nCiudades por región:")
print(df['region'].value_counts())
print(f"\nValores nulos por columna:")
print(df.isnull().sum())
print(f"\nEstadísticas descriptivas:")
df.describe().round(2)

Total ciudades: 578
Países únicos: 127
Regiones únicas: 7

Ciudades por región:
region
Europe         210
Americas       142
Other          131
Asia            63
Oceania         15
Middle East     13
Africa           4
Name: count, dtype: int64

Valores nulos por columna:
city_name                   0
country                     0
region                      0
average_salary              0
average_rent                0
tax_rate                    0
cost_of_living_index        0
purchasing_power_index      0
quality_of_life_index       0
job_market_score            0
net_salary                  0
purchasing_power            0
annual_savings              0
relocation_score            0
lat                       254
lon                       254
dtype: int64

Estadísticas descriptivas:


,average_salary,average_rent,tax_rate,cost_of_living_index,purchasing_power_index,quality_of_life_index,job_market_score,net_salary,purchasing_power,annual_savings,relocation_score,lat,lon
count,578.00,578.00,578.00,578.00,578.00,578.00,578.00,578.00,578.00,578.00,578.00,324.00,324.00
mean,53263.41,858.16,45.11,57.54,71.50,64.86,40.78,28983.32,519.53,8695.00,58.84,37.93,21.25
std,26749.17,574.49,5.44,21.66,34.21,20.43,19.96,14363.96,229.05,4309.19,20.29,22.96,55.87
min,7774.00,76.00,15.00,18.55,1.62,23.00,0.00,4275.70,102.01,1282.71,17.24,-43.53,-123.35
25%,38443.00,395.00,43.60,38.02,42.76,47.60,24.02,21219.55,399.24,6365.86,41.57,32.52,-0.37
50%,52107.00,749.50,45.00,62.40,70.94,65.35,40.45,29351.30,480.33,8805.39,59.34,45.90,12.17
75%,60286.00,1179.25,47.00,73.03,95.68,78.40,54.88,33786.60,599.05,10135.98,72.54,51.45,34.77
max,358924.00,3491.00,55.90,149.02,172.98,126.30,100.00,197408.20,3852.62,59222.46,119.75,65.02,176.17


## Distribución del Relocation Score


In [3]:
import plotly.express as px
import plotly.graph_objects as go

# Histograma del Relocation Score
fig = px.histogram(
    df, x="relocation_score",
    nbins=30,
    title="Distribución del Relocation Score",
    labels={"relocation_score": "Relocation Score", "count": "Número de ciudades"},
    color_discrete_sequence=["#f5f5f7"],
    template="plotly_dark",
)
fig.update_layout(
    paper_bgcolor="#111111",
    plot_bgcolor="#1c1c1e",
    showlegend=False,
)
fig.show()

# Estadísticas clave
print(f"Mínimo: {df['relocation_score'].min():.1f}")
print(f"Máximo: {df['relocation_score'].max():.1f}")
print(f"Media: {df['relocation_score'].mean():.1f}")
print(f"Mediana: {df['relocation_score'].median():.1f}")
print(f"\nTop 5 ciudades:")
print(df.nlargest(5, 'relocation_score')[['city_name','country','relocation_score']])
print(f"\nBottom 5 ciudades:")
print(df.nsmallest(5, 'relocation_score')[['city_name','country','relocation_score']])

Mínimo: 17.2
Máximo: 119.8
Media: 58.8
Mediana: 59.3

Top 5 ciudades:
         city_name        country  relocation_score
276    Houston, TX  United States            119.75
233     Dallas, TX  United States            118.38
190  Ann Arbor, MI  United States            112.03
250     Austin, TX  United States            110.97
130   San Jose, CA  United States            110.47

Bottom 5 ciudades:
       city_name      country  relocation_score
315       Havana         Cuba             17.24
454     Damascus        Syria             19.26
367      Abidjan  Ivory Coast             20.01
443        Lagos      Nigeria             21.21
348  Addis Ababa     Ethiopia             22.12


## Distribución del coste de vida

In [4]:
fig = px.box(
    df,
    x="region",
    y="cost_of_living_index",
    title="Distribución del coste de vida por región",
    labels={
        "cost_of_living_index": "Índice de coste de vida",
        "region": "Región"
    },
    color="region",
    color_discrete_sequence=px.colors.sequential.Greys_r,
    template="plotly_dark",
)
fig.update_layout(
    paper_bgcolor="#111111",
    plot_bgcolor="#1c1c1e",
    showlegend=False,
)
fig.show()

print(df.groupby('region')['cost_of_living_index'].agg(['mean','median','min','max']).round(1))

             mean  median   min    max
region                                
Africa       42.2    41.9  40.3   44.9
Americas     67.6    70.4  23.7  103.6
Asia         34.7    26.1  20.8   85.6
Europe       65.8    67.6  34.5  131.2
Middle East  68.3    62.9  48.4   94.5
Oceania      76.9    76.6  70.9   83.2
Other        41.5    37.5  18.6  149.0


## Mapa de calor de correlaciones

In [5]:
import plotly.figure_factory as ff
import numpy as np

cols_corr = [
    'relocation_score', 'purchasing_power_index', 'quality_of_life_index',
    'job_market_score', 'cost_of_living_index', 'tax_rate',
    'average_rent', 'average_salary'
]

corr_matrix = df[cols_corr].corr().round(2)

fig = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.columns,
    colorscale='RdBu',
    zmid=0,
    text=corr_matrix.values.round(2),
    texttemplate="%{text}",
    textfont={"size": 10},
))
fig.update_layout(
    title="Mapa de calor de correlaciones entre indicadores",
    paper_bgcolor="#111111",
    plot_bgcolor="#1c1c1e",
    font=dict(color="#f5f5f7"),
    width=700, height=600,
)
fig.show()

# Correlaciones más fuertes
print("Correlaciones más relevantes con Relocation Score:")
print(corr_matrix['relocation_score'].sort_values(ascending=False))

Correlaciones más relevantes con Relocation Score:
relocation_score          1.00
purchasing_power_index    1.00
quality_of_life_index     1.00
job_market_score          1.00
cost_of_living_index      0.68
average_rent              0.67
average_salary            0.46
tax_rate                  0.02
Name: relocation_score, dtype: float64


## Distribución de salarios por región

In [6]:
df_sal = pd.read_csv('../data/processed/salaries.csv')

print(f"Total registros: {len(df_sal)}")
print(f"Países únicos: {df_sal['city_name'].nunique()}")
print(f"Roles únicos: {df_sal['role'].nunique()}")
print(f"\nEstadísticas de salario:")
print(df_sal['gross_salary'].describe().round(0))

# Top y bottom países por salario mediano
medians = df_sal.groupby('city_name')['gross_salary'].median().sort_values(ascending=False)
print(f"\nTop 5 países por salario mediano:")
print(medians.head(5))
print(f"\nBottom 5 países por salario mediano:")
print(medians.tail(5))

# Distribución por rol
fig = px.box(
    df_sal,
    x="specialization",
    y="gross_salary",
    title="Distribución salarial por especialización",
    labels={
        "gross_salary": "Salario bruto anual (EUR)",
        "specialization": "Especialización"
    },
    color_discrete_sequence=["#f5f5f7"],
    template="plotly_dark",
)
fig.update_layout(paper_bgcolor="#111111", plot_bgcolor="#1c1c1e")
fig.show()

Total registros: 18112
Países únicos: 149
Roles únicos: 32

Estadísticas de salario:
count     18112.0
mean      89258.0
std       63461.0
min        4605.0
25%       45761.0
50%       75150.0
75%      117407.0
max      459770.0
Name: gross_salary, dtype: float64

Top 5 países por salario mediano:
city_name
North Korea                 408893.0
Oman                        358924.0
Andorra                     184115.0
United States of America    138000.0
Switzerland                 131185.0
Name: gross_salary, dtype: float64

Bottom 5 países por salario mediano:
city_name
Djibouti                       5366.0
Libyan Arab Jamahiriya         5098.0
United Republic of Tanzania    4888.5
Somalia                        4876.0
Angola                         4815.0
Name: gross_salary, dtype: float64
